In [ ]:
#!/usr/bin/env python
# coding: utf-8

# # Bayesian Non-parametric Causal Inference

# In[1]:


import arviz as az
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from scipy.stats import norm
import pandas as pd
import geopandas as gpd
import pymc as pm
import pymc_bart as pmb
import nutpie
import pytensor.tensor as pt
import os


# # Estimating Treatment Effects
# 
# In this section we build a set of functions to pull out and extract a sample from our posterior distribution of propensity scores; use this propensity score to reweight the observed outcome variable across our treatment and control groups to re-calculate the average treatment effect (ATE). It reweights our data using one of three inverse probability weighting schemes and then plots three views (1) the raw propensity scores across groups (2) the raw outcome distribution and (3) the re-weighted outcome distribution.
# 
# First we define a bunch of helper functions for each weighting adjustment method we will explore:

# In[3]:


# We need to do this process once for every iteration x chain (i.e., thousands of times), so we'll use statsmodel OLS rather than a complex Bayesian model like BART
def fit_outcome_model(X, predictors, y, model):
    cbsa, _  = X.cbsa.factorize()
    cbg, _  = X.geoid10.factorize()
    coords = {"cbg_id": cbg, "cbsa_id":cbsa}
    
    X1 = X.loc[:, predictors]

    with pm.Model(coords=coords) as bart_model:

        cbsa_id = pm.Data("cbsa_id", cbsa, dims="cbg_id")

        # # Linear parameters for all variables included in the BART portion
        # beta = pm.Normal("beta", mu=0, sigma=1, shape=X1.shape[1])
        # linear_predictor = pm.math.dot(X1, beta)  # Linear combination
        
        w_mean = pmb.BART("w_mean", X=X1, Y=y, m=25)
        
        # # Combine BART and linear predictor
        # combined_predictor = w_mean + linear_predictor
        
        sigma_y = pm.HalfCauchy("sigma_y", beta=10)
        
        # Independent random intercepts
        mu_a = pm.Normal("mu_a", 8, 5)
        sigma_a = pm.Exponential("sigma_a", 2)
        z_a = pm.Normal("z_a", mu=0, sigma=1, dims="cbsa_id")
        alpha = pm.Deterministic("alpha", mu_a + z_a * sigma_a, dims="cbsa_id")

        # Expected value
        y_hat = pm.Deterministic('y_hat', w_mean+alpha, dims="cbg_id")
        
        # likelihood of the observed data
        y_i = pm.Normal("y_i", mu=y_hat, sigma=sigma_y, observed=y, dims="cbg_id")
        
    compiled_model = nutpie.compile_pymc_model(bart_model)
    # trace = nutpie.sample(compiled_model, chains=4, tune=10, draws=10)
    trace = nutpie.sample(compiled_model, chains=4)
    trace.to_netcdf("/home/mehrnoosh.zare/vulcan-results/vulcan-{0}-pie.nc".format(model))
    return trace

def make_doubly_robust_adjustment(y_pred, y, i, ips):
    
    dr_estimate = y_pred + ips * (y - y_pred)

    return dr_estimate

def get_ate(trace, t, y, ips, i):
    ### Post processing the sample posterior distribution for propensity scores
    ### One sample at a time.
    y_pred = trace["posterior"]["y_hat"].stack(z=("chain", "draw"))[:, i].values

    dr_estimate = make_doubly_robust_adjustment(
        y_pred, y, i, ips
    )
    
    bandwidth = 0.025 * (np.max(t) - np.min(t))  # Example: 5% of range
    
    t_high, t_low = np.percentile(t, [75, 25])
    
    trt = np.mean(dr_estimate[(t >= t_high - bandwidth) & (t <= t_high + bandwidth)])
    ntrt = np.mean(dr_estimate[(t >= t_low - bandwidth) & (t <= t_low + bandwidth)])
    
    ate = np.exp(trt) - np.exp(ntrt)
    if i%500==0:
        print("Done", i)
        # print("trt",trt)
        # print("ntrt",ntrt)
    return [ate, trt, ntrt]


# Next, and because we are Bayesians - we pull out and evaluate the posterior distribution of the ATE based on the sampled propensity scores. We’ve seen a point estimate for the ATE above, but it’s often more important in the causal inference context to understand the uncertainty in the estimate.

# In[4]:


def plot_ate(ate_dist_df, model, xy=(4.0, 250)):
    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    #axs = axs.flatten()
    ax.hist(
        ate_dist_df["ATE"], bins=30, ec="black", color="slateblue", label="ATE", alpha=0.6
    )
    #axs[0].hist(
        #ate_dist_df["E(Y(25per))"], bins=30, ec="black", color="red", label="E(Y(25per))", alpha=0.7
    #)
    #axs[1].hist(ate_dist_df["ATE"], bins=30, ec="black", color="slateblue", label="ATE", alpha=0.6)
    ate = np.round(ate_dist_df["ATE"].mean(), 2)
    ax.axvline(ate, label="E(ATE)", linestyle="--", color="black")
    ax.annotate(f"E(ATE): {ate}", xy, fontsize=20, fontweight="bold")
    ax.set_title(f"Average Treatment Effect \n E(ATE): {ate}", fontsize=20)
    #axs[0].set_title("E(Y) Distributions for Treated and Control", fontsize=20)
    ax.set_xlabel("Average Treatment Effect")
    #axs[0].set_xlabel("Expected Potential Outcomes")
    #axs[1].legend()
    ax.legend()
    plt.savefig("/home/mehrnoosh.zare/vulcan-results/vulcan-{0}-pie.pdf".format(model), format="pdf")
    plt.close(fig)  # Closes the figure to free memory


# # Transportation

# In[5]:


tran_par_file = '/work/hawkins_lab/vulcan/data/vulcan/parquet/vulcan_ONR_epa.parquet'
tran_df = gpd.read_parquet(tran_par_file)
tran_wgt_file = '/work/hawkins_lab/vulcan/results/tran_weights.csv'
tran_wgt_df = pd.read_csv(tran_wgt_file)
# Create columns that assign decile numbers for each treatment variable
tran_df['treat_diversity'] = tran_df['d2b_e5mixa']


# In[6]:


tran_df = pd.concat([tran_df,tran_wgt_df], axis=1)

tran_df = tran_df[(tran_df["value_weig"]>0) & (tran_df["totpop"]>0)]


# TEMP!!!!

# In[7]:


def weighted_avg(values, weights):
    return (values * weights.sum(axis=1)).sum() / weights.sum(axis=1).sum()

grp_col = "cbsa"
val_col = "d4a"
agg_val_col = "d4a_cbsa"

tran_df_cbsa = tran_df.drop_duplicates(subset=grp_col).copy()

wgt_cols = ["totpop"]
tran_df_cbsa.loc[:, val_col] = tran_df.groupby(grp_col).apply(lambda x: weighted_avg(x[val_col].replace(-99999,1), x.loc[:, wgt_cols]), include_groups=False # Use 1 not 0 for ln()
    ).values

d4a_dict = dict(zip(tran_df_cbsa[grp_col], tran_df_cbsa[val_col]))
tran_df[agg_val_col] = tran_df[grp_col].map(d4a_dict) # Need to use a merging method


# In[8]:


predictors = ["totpop_cbsa",
              "d1aa_cbsa",
            # "d1a_cbsa"]
             "d2b_e5mix_cbsa"]
            #   "d3a_cbsa",
            #  "d4a_cbsa", 
            #  "d5ar_cbsa"]

X = tran_df.copy()
    
X["d1aa_cbsa"] = tran_df["totpop_cbsa"]/tran_df["ac_unpr_cbsa"] # true cbsa density

y = tran_df["value_weig"] / tran_df["totpop"].replace(0, np.nan) # make per capita

X.loc[:,predictors] = np.log(X.loc[:,predictors])
predictors+=["ps"]
y = np.log(y)


# In[9]:


y.isna().sum()


# ## Diversity

# In[ ]:


X["ps"] = X["div"]
ips = 1/X["ps"]
t_diversity = tran_df["treat_diversity"]

trace = fit_outcome_model(X, predictors, y, "tran_div")

qs = range(4000)
ate_dist = [get_ate(trace, t_diversity, y, ips, q) for q in qs]

ate_dist_df = pd.DataFrame(ate_dist, columns=["ATE", "E(Y(75per))", "E(Y(25per))"])
ate_dist_df.head()


# In[ ]:


plot_ate(ate_dist_df, "tran_div")


Progress,Draws,Divergences,Step Size,Gradients/Draw
,1400,0,0.06,63
,283,0,0.01,1023
,320,0,0.01,1023
,269,0,0.01,1023
